In [1]:
import re
import csv
from pathlib import Path
from collections import defaultdict, Counter

In [2]:
CODE_DIR = 'dataset/storybook/src/code'
CODE_DIR_PATH = Path(CODE_DIR)

# 'components' or 'uis'
TYPE = 'components'
TYPE_DIR_PATH = CODE_DIR_PATH / TYPE

COMPLEXITIES = ['simple', 'medium', 'hard']

# Directory structure
#   {TYPE}/gt/{complexity}/{file}.vue
#   {TYPE}/{a|b|c|d}/{prompt_strategy}/{complexity}/{file}.vue
APPROACHES = ['gt', 'a', 'b', 'c', 'd']

PRIMEVUE_COMPONENTS = {
    'Accordion', 'AccordionPanel', 'AccordionHeader', 'AccordionContent',
    'Avatar', 'AvatarGroup',
    'Badge', 'Breadcrumb', 'Button',
    'Card', 'Checkbox', 'Column', 'ColumnGroup',
    'DataTable', 'DatePicker', 'Dialog', 'Divider',
    'InputNumber', 'InputText',
    'Menu',
    'OverlayBadge',
    'Password', 'Popover', 'ProgressBar',
    'RadioButton', 'Row',
    'Select', 'Skeleton', 'Slider',
    'Tab', 'TabList', 'TabPanel', 'TabPanels', 'Tabs', 'Tag', 'Textarea',
    'ToggleSwitch',
    'IconField', 'InputIcon',
}

PRIMEVUE_COMPONENTS_NORM = {c.lower(): c for c in PRIMEVUE_COMPONENTS}

# 'class'/'style'/'id'/'ref'/'key' sind keine PrimeVue-Props. Directiven (v-if, v-for, ...)
# werden generisch ueber startswith('v-') ausgeschlossen; v-model wird NICHT hier gefiltert,
# sondern ueber eine eigene Regel in parse_props() als Prop normalisiert (siehe unten).
SKIP_PROPS = {'class', 'style', 'id', 'ref', 'key'}

## Load reference and generated code files in groups by method, prompt strategy and complexity

In [3]:
CODE_FILES_BY_APPROACH: dict = {
    'gt': defaultdict(dict),
    'a':  defaultdict(lambda: defaultdict(dict)),
    'b':  defaultdict(lambda: defaultdict(dict)),
    'c':  defaultdict(lambda: defaultdict(dict)),
    'd':  defaultdict(lambda: defaultdict(dict)),
}

for approach in APPROACHES:
    approach_dir = TYPE_DIR_PATH / approach

    print(f'Loading code files for approach {approach} from {approach_dir}')

    if not approach_dir.exists():
        print(f'    Skip (not existing): {approach_dir}')

        continue

    if approach == 'gt':
        # gt/{complexity}/{file}.vue
        for complexity_dir in approach_dir.iterdir():
            if not complexity_dir.is_dir() or complexity_dir.name not in COMPLEXITIES:
                continue

            complexity = complexity_dir.name

            for vue_file in complexity_dir.glob('*.vue'):
                content = vue_file.read_text(encoding='utf-8', errors='ignore')

                if vue_file.stem in CODE_FILES_BY_APPROACH['gt'][complexity]:
                    print(f'  WARNING: Duplicate file {vue_file.stem} in {complexity} for gt, overwriting previous content.')

                CODE_FILES_BY_APPROACH['gt'][complexity][vue_file.stem] = content

    else:
        # {approach}/{prompt_strategy}/{complexity}/{file}.vue
        for strategy_dir in approach_dir.iterdir():

            if not strategy_dir.is_dir():
                continue

            prompt_strategy = strategy_dir.name

            for complexity_dir in strategy_dir.iterdir():
                if not complexity_dir.is_dir() or complexity_dir.name not in COMPLEXITIES:
                    continue

                complexity = complexity_dir.name

                for vue_file in complexity_dir.glob('*.vue'):
                    content = vue_file.read_text(encoding='utf-8', errors='ignore')

                    if vue_file.stem in CODE_FILES_BY_APPROACH[approach][prompt_strategy][complexity]:
                        print(f'  WARNING: Duplicate file {vue_file.stem} in {complexity} for {approach}/{prompt_strategy}, overwriting previous content.')

                    CODE_FILES_BY_APPROACH[approach][prompt_strategy][complexity][vue_file.stem] = content

print('\nCode files loaded:')
for approach, data in CODE_FILES_BY_APPROACH.items():

    if approach == 'gt':
        total_files = sum(len(files) for files in data.values())

        print(f'Approach {approach}: {total_files} files')

    else:
        total_files = sum(len(files) for strategies in data.values() for files in strategies.values())

        print(f'Approach {approach}: {total_files} files')

        for strategy, complexities in data.items():
            total_files = sum(len(files) for files in complexities.values())

            print(f'  Strategy {strategy}: {total_files} files')

            for complexity, files in complexities.items():
                print(f'    Complexity {complexity}: {len(files)} files')

Loading code files for approach gt from dataset\storybook\src\code\components\gt
Loading code files for approach a from dataset\storybook\src\code\components\a
Loading code files for approach b from dataset\storybook\src\code\components\b
Loading code files for approach c from dataset\storybook\src\code\components\c
Loading code files for approach d from dataset\storybook\src\code\components\d

Code files loaded:
Approach gt: 30 files
Approach a: 0 files
Approach b: 540 files
  Strategy few_shot: 270 files
    Complexity hard: 90 files
    Complexity medium: 90 files
    Complexity simple: 90 files
  Strategy zero_shot: 270 files
    Complexity hard: 90 files
    Complexity medium: 90 files
    Complexity simple: 90 files
Approach c: 540 files
  Strategy few_shot: 270 files
    Complexity hard: 90 files
    Complexity medium: 90 files
    Complexity simple: 90 files
  Strategy zero_shot: 270 files
    Complexity hard: 90 files
    Complexity medium: 90 files
    Complexity simple: 90 f

## Component-Prop Accuracy

In [4]:
def extract_template_block(sfc: str) -> str | None:
    """Finds the outer <template> block to be deeply robust against nested
    named-slot templates (<template #slotname>...</template>), which frequently occur in PrimeVue SFCs
    (icon slots, card headers, DataTable column bodies, ...).

    A non-greedy regex ‘<template>(.*?)</template>’ would stop at the FIRST
    closing </template>—that is, at a nested slot,
    not at the actual end of the SFC template—and thereby silently omit all
    subsequent components.
    """
    start = re.search(r'<template>', sfc)

    if not start:
        return None

    pos = start.end()
    depth = 1

    for m in re.finditer(r'<template(?:\s[^>]*)?(/?)>|</template>', sfc[pos:]):
        token = m.group(0)

        if token == '</template>':
            depth -= 1

            if depth == 0:
                return sfc[pos: pos + m.start()]
        elif not token.endswith('/>'):
            depth += 1   # nested, non-self-closing <template ...>

    return sfc[pos:]  # Fallback if no end is found (should not happen)


_LITERAL_RE = re.compile(r"^\s*(true|false|-?\d+(\.\d+)?|'[^']*'|\"[^\"]*\")\s*$", re.IGNORECASE)


def is_literal_expr(expr: str) -> bool:
    """Prueft, ob ein dynamischer Prop-Ausdruck (':prop=\"expr\"') ein Literal ist
    (Zahl, Boolean, String) statt einer Variablen-/Ausdrucksreferenz. Nur Literale
    sind auf Quelltextebene sinnvoll vergleichbar -- ':severity=\"statusSeverity\"'
    referenziert eine lokale Variable, deren Name zwischen Modellen unabhaengig
    vergeben wird und die deshalb NICHT direkt verglichen werden darf.
    """
    return bool(_LITERAL_RE.match(expr.strip()))


def normalize_prop_value(raw: str) -> str:
    """Normalisiert Anfuehrungszeichen, Gross/Kleinschreibung bei Booleans und
    numerische Formatierung, damit rein syntaktische Abweichungen nicht als
    inhaltlicher Fehler gewertet werden."""
    if raw == '__boolean__':
        return 'true'

    v = raw.strip()

    if (v.startswith('"') and v.endswith('"')) or (v.startswith("'") and v.endswith("'")):
        v = v[1:-1]

    low = v.lower()
    if low in ('true', 'false'):
        return low

    try:
        return repr(float(v))
    except ValueError:
        pass

    return v


def parse_props(attrs_str: str) -> dict[str, str]:
    """Extracts semantic props from the attribute string of a Vue tag.

    Rueckgabeschluessel:
      'prop'   -- statischer Prop (prop="wert"), Wert ist bereits ein Literal
      ':prop'  -- dynamischer Prop (:prop="expr"), Wert kann Literal ODER
                  Variablenreferenz sein (siehe is_literal_expr)
    v-model / v-model:name werden auf ':modelValue' bzw. ':<name>' normalisiert,
    da PrimeVue-Komponenten intern ueber modelValue/update:modelValue binden.
    """
    props: dict[str, str] = {}

    if not attrs_str:
        return props

    # 0. v-model / v-model:propName -- als dynamischer Prop behandelt
    for m in re.finditer(r'v-model(?::([a-zA-Z][\w-]*))?="([^"]*)"', attrs_str):
        named_prop, val = m.group(1), m.group(2)
        key = f':{named_prop}' if named_prop else ':modelValue'
        props[key] = val

    # 1. Dynamic Props: :prop="value"
    for m in re.finditer(r':([a-zA-Z][\w-]*)="([^"]*)"', attrs_str):
        key, val = m.group(1), m.group(2)

        if key not in SKIP_PROPS and not key.startswith('v-'):
            props[f':{key}'] = val

    # 2. Static Props: prop="value"
    for m in re.finditer(r'(?<!:)\b([a-zA-Z][\w-]*)="([^"]*)"', attrs_str):
        key, val = m.group(1), m.group(2)

        if key not in SKIP_PROPS and not key.startswith(('v-', '@')):
            props[key] = val

    # 3. Boolean Props without Value: binary, showButtons, toggleMask
    # WICHTIG: alle 'key="value"' / ':key="value"' / 'v-model[:x]="value"' / '@key="value"'
    # Spans zuerst entfernen -- sonst werden Woerter INNERHALB von Attributwerten
    # (z.B. 'btnColor' in :severity="btnColor", oder 'true'/'false'/Variablennamen)
    # faelschlich als eigenstaendige Boolean-Props erkannt.
    stripped = re.sub(r'(?:@|:)?[a-zA-Z][\w-]*(?::[a-zA-Z][\w-]*)?="[^"]*"', ' ', attrs_str)

    for m in re.finditer(r'\b([a-zA-Z][\w-]*)\b', stripped):
        key = m.group(1)

        if key not in SKIP_PROPS and not key.startswith('v-') \
           and key not in props and key[0].islower():
            props[key] = '__boolean__'

    return props


_TAG_RE = re.compile(r'<([A-Za-z][A-Za-z0-9]*)((?:\s[^>]*?)?)\s*(/?)>')


def extract_component_instances(sfc: str) -> dict[str, list[dict[str, str]]]:
    """Extracts all instances of known PrimeVue components together with their props,
    in document order (Grundlage der ordnungsbasierten Instanzpaarung, s. evaluation.md).

    Ein einziger Scan ueber alle Tags statt eines Regex pro bekannter Komponente:
    schneller UND case-insensitiv (ueber PRIMEVUE_COMPONENTS_NORM), sodass z.B.
    '<button>' korrekt als 'Button' erkannt wird -- identisches Prinzip wie in
    extract_components_detailed() aus evaluation_f1_scores.ipynb.

    Returns: {'Button': [{'label': 'OK', ':severity': 'warn'}, ...], ...}
    """
    template = extract_template_block(sfc)
    instances: dict[str, list[dict[str, str]]] = {}

    if template is None:
        return instances

    for m in _TAG_RE.finditer(template):
        tag, attrs_str = m.group(1), m.group(2)
        canonical = PRIMEVUE_COMPONENTS_NORM.get(tag.lower())

        if canonical is None:
            continue

        instances.setdefault(canonical, []).append(parse_props(attrs_str))

    return instances

In [5]:
def compute_prop_accuracy(
    gen_instances: dict[str, list[dict]],
    gt_instances:  dict[str, list[dict]],
) -> dict:
    """Vergleicht Props gepaarter Komponenteninstanzen (True Positives aus Component-F1).

    Instanzpaarung: k-tes Vorkommen in gen_instances mit k-tem Vorkommen in
    gt_instances (Dokumentreihenfolge, via zip() -- ueberzaehlige Instanzen werden
    NICHT gepaart, da sie bereits als FP/FN in Component-F1 eingehen).

    Kategorien pro Prop:
      correct      -- Name und (normalisierter) Wert stimmen ueberein
      wrong        -- Name stimmt, Wert weicht ab
      hallucinated -- Prop im Generat, aber nicht in der Ground Truth
      missing      -- Prop in der Ground Truth, aber nicht im Generat
      unresolved   -- dynamischer Prop, dessen Wert (bei mind. einer Seite) keine
                      literale Konstante ist, sondern eine Variablenreferenz
                      (':severity="statusSeverity"') -- auf Quelltextebene nicht
                      sicher vergleichbar, wird deshalb WEDER als correct NOCH
                      als wrong gezaehlt.

    Zwei getrennte Kennzahlen (siehe evaluation.md):
      prop_accuracy = correct / (correct + wrong + hallucinated)   -- OHNE missing
      missing_rate  = missing / (missing + correct + wrong)        -- Vollstaendigkeit

    Mockups ohne vergleichbare Props (Nenner = 0) liefern None statt eines
    irrefuehrenden Default-Werts -- werden in der Aggregation ausgeschlossen,
    nicht als perfekt oder als 0 gewertet.
    """
    per_component = {}
    total_correct = total_wrong = total_hallucinated = total_missing = total_unresolved = 0

    # Nur True Positives (Komponente in beiden vorhanden)
    common = set(gen_instances.keys()) & set(gt_instances.keys())

    for comp in common:
        gen_list = gen_instances[comp]
        gt_list  = gt_instances[comp]

        correct = wrong = hallucinated = missing = unresolved = 0

        for gen_props, gt_props in zip(gen_list, gt_list):
            for key, gt_val in gt_props.items():
                if key not in gen_props:
                    missing += 1
                    continue

                gen_val = gen_props[key]
                dynamic = key.startswith(':')

                if dynamic and not (is_literal_expr(gen_val) and is_literal_expr(gt_val)):
                    unresolved += 1
                elif normalize_prop_value(gen_val) == normalize_prop_value(gt_val):
                    correct += 1
                else:
                    wrong += 1

            for key in gen_props:
                if key not in gt_props:
                    hallucinated += 1

        acc_denom = correct + wrong + hallucinated
        per_component[comp] = {
            'accuracy':     round(correct / acc_denom, 4) if acc_denom else None,
            'correct':      correct,
            'wrong':        wrong,
            'hallucinated': hallucinated,
            'missing':      missing,
            'unresolved':   unresolved,
            'instances':    min(len(gen_list), len(gt_list)),
        }

        total_correct      += correct
        total_wrong         += wrong
        total_hallucinated += hallucinated
        total_missing       += missing
        total_unresolved    += unresolved

    prop_acc_denom = total_correct + total_wrong + total_hallucinated
    missing_denom  = total_missing + total_correct + total_wrong

    return {
        'prop_accuracy': round(total_correct / prop_acc_denom, 4) if prop_acc_denom else None,
        'missing_rate':  round(total_missing / missing_denom, 4) if missing_denom else None,
        'per_component': per_component,
        'totals': {
            'correct':      total_correct,
            'wrong':        total_wrong,
            'hallucinated': total_hallucinated,
            'missing':      total_missing,
            'unresolved':   total_unresolved,
        },
    }

In [6]:
def parse_generated_stem(stem: str, approach: str) -> dict | None:
    """Parses filenames such as:
      ‘1-a’                          -> Approach A (deterministic, no model/run)
      '1-b2-claude-sonnet-5-1'       -> Approach B, Strategy b2, Model, Run

    Returns: {‘index’, ‘strategy’, ‘model’, ‘run’} or None if the pattern
    does not match (the approach must already be known—it comes from the directory level).
    """
    parts = stem.split('-')

    if not parts[0].isdigit():
        return None

    index = parts[0].zfill(2)

    if approach == 'a':
        if len(parts) == 2 and parts[1] == 'a':
            return {'index': index, 'strategy': 'a', 'model': None, 'run': None}

        return None

    if len(parts) >= 4 and re.match(rf'^{approach}[123]$', parts[1]) and parts[-1].isdigit():
        strategy = parts[1]
        model = '-'.join(parts[2:-1])
        run = parts[-1]

        return {'index': index, 'strategy': strategy, 'model': model, 'run': run}

    return None


def gt_index(stem: str) -> str:
    """Normalizes a GT stem (‘1’, ‘01’, ...) to the 2-digit index."""
    return stem.zfill(2) if stem.isdigit() else stem


def format_counter(c: Counter) -> str:
    """'Button:2;Card:1' for CSV storage; leave blank instead of ‘{}’ if the counter is empty."""
    return ';'.join(f'{k}:{v}' for k, v in sorted(c.items()))

In [7]:
prop_results: list[dict] = []

for complexity in COMPLEXITIES:
    gt_files = CODE_FILES_BY_APPROACH['gt'].get(complexity, {})
    gt_by_index = {gt_index(stem): content for stem, content in gt_files.items()}
    gt_instances_by_index = {idx: extract_component_instances(sfc) for idx, sfc in gt_by_index.items()}

    for approach in ['a', 'b', 'c', 'd']:
        approach_data = CODE_FILES_BY_APPROACH[approach]

        for prompt_strategy, by_complexity in approach_data.items():
            gen_files = by_complexity.get(complexity, {})

            for stem, gen_sfc in gen_files.items():
                parsed = parse_generated_stem(stem, approach)

                if parsed is None:
                    print(f'   WARNING: Filename does not match expected pattern: {approach}/{prompt_strategy}/{complexity}/{stem}')

                    continue

                gt_instances = gt_instances_by_index.get(parsed['index'])

                if gt_instances is None:
                    print(f'  WARNING: No GT found for {approach}/{prompt_strategy}/{complexity}/{stem} (Index {parsed["index"]})')

                    continue

                gen_instances = extract_component_instances(gen_sfc)
                scores = compute_prop_accuracy(gen_instances, gt_instances)

                per_comp_str = format_counter(Counter({
                    comp: v['wrong'] + v['hallucinated']
                    for comp, v in scores['per_component'].items()
                    if v['wrong'] + v['hallucinated'] > 0
                }))

                acc_str = f'{scores["prop_accuracy"]:.2f}' if scores['prop_accuracy'] is not None else '-'
                mr_str  = f'{scores["missing_rate"]:.2f}'  if scores['missing_rate']  is not None else '-'

                print(f'{complexity}/{parsed["index"]} '
                      f'[{approach}/{prompt_strategy}/{parsed["strategy"]}/{parsed["model"] or "-"}]  '
                      f'PropAcc={acc_str}  MissingRate={mr_str}')

                prop_results.append({
                    'mockup':          f'{complexity}-{parsed["index"]}',
                    'complexity':       complexity,
                    'index':            parsed['index'],
                    'approach':         approach,
                    'strategy':         parsed['strategy'],
                    'prompt_strategy':  prompt_strategy,
                    'model':            parsed['model'] or '',
                    'run':              parsed['run'] or '',
                    'prop_accuracy':    scores['prop_accuracy'],
                    'missing_rate':     scores['missing_rate'],
                    'correct':          scores['totals']['correct'],
                    'wrong':            scores['totals']['wrong'],
                    'hallucinated':     scores['totals']['hallucinated'],
                    'missing':          scores['totals']['missing'],
                    'unresolved':       scores['totals']['unresolved'],
                    'error_components': per_comp_str,
                })

print(f'\nComputed: {len(prop_results)} Prop-Accuracy-Scores')

simple/01 [b/few_shot/b1/claude-sonnet-5]  PropAcc=0.80  MissingRate=0.17
simple/01 [b/few_shot/b1/gemini-3.1-pro-preview]  PropAcc=1.00  MissingRate=0.17
simple/01 [b/few_shot/b1/gpt-5.6-terra]  PropAcc=0.83  MissingRate=0.17
simple/01 [b/few_shot/b2/claude-sonnet-5]  PropAcc=0.71  MissingRate=0.17
simple/01 [b/few_shot/b2/gemini-3.1-pro-preview]  PropAcc=0.57  MissingRate=0.33
simple/01 [b/few_shot/b2/gpt-5.6-terra]  PropAcc=0.33  MissingRate=0.50
simple/01 [b/few_shot/b3/claude-sonnet-5]  PropAcc=0.83  MissingRate=0.17
simple/01 [b/few_shot/b3/gemini-3.1-pro-preview]  PropAcc=0.83  MissingRate=0.17
simple/01 [b/few_shot/b3/gpt-5.6-terra]  PropAcc=0.50  MissingRate=0.33
simple/10 [b/few_shot/b1/claude-sonnet-5]  PropAcc=0.54  MissingRate=0.16
simple/10 [b/few_shot/b1/gemini-3.1-pro-preview]  PropAcc=0.35  MissingRate=0.56
simple/10 [b/few_shot/b1/gpt-5.6-terra]  PropAcc=0.43  MissingRate=0.32
simple/10 [b/few_shot/b2/claude-sonnet-5]  PropAcc=0.55  MissingRate=0.20
simple/10 [b/few_s

In [8]:
def aggregate_prop_macro(items: list[dict]) -> dict:
    """Makro-Mittel: Durchschnitt der Pro-Mockup-Werte. Mockups ohne vergleichbare
    Props (prop_accuracy/missing_rate = None) werden von der jeweiligen Aggregation
    ausgeschlossen, nicht als 0 oder 1 gewertet."""
    resolved_acc = [i['prop_accuracy'] for i in items if i['prop_accuracy'] is not None]
    resolved_mr  = [i['missing_rate']  for i in items if i['missing_rate']  is not None]

    return {
        'prop_accuracy_macro': round(sum(resolved_acc) / len(resolved_acc), 4) if resolved_acc else None,
        'missing_rate_macro':  round(sum(resolved_mr) / len(resolved_mr), 4) if resolved_mr else None,
        'n_resolved_acc': len(resolved_acc),
        'n_resolved_mr':  len(resolved_mr),
        'n': len(items),
    }


def aggregate_prop_micro(items: list[dict]) -> dict:
    """Mikro-Mittel: globale Summen zuerst, dann EINE Division."""
    correct      = sum(i['correct']      for i in items)
    wrong        = sum(i['wrong']        for i in items)
    hallucinated = sum(i['hallucinated'] for i in items)
    missing      = sum(i['missing']      for i in items)

    acc_denom = correct + wrong + hallucinated
    mr_denom  = missing + correct + wrong

    return {
        'prop_accuracy_micro': round(correct / acc_denom, 4) if acc_denom else None,
        'missing_rate_micro':  round(missing / mr_denom, 4) if mr_denom else None,
    }


def group_key(r: dict) -> tuple:
    """Grouping by approach, strategy, model, AND prompt strategy ->
    Zero-Shot and Few-Shot must not be included in the same mean, since both
    are different independent variables in the design."""
    return (r['approach'], r['strategy'], r['model'], r['prompt_strategy'])


def group_label(key: tuple) -> str:
    approach, strategy, model, prompt_strategy = key
    base = strategy if not model else f'{strategy}_{model}'

    return f'{base}_{prompt_strategy}'


by_method_prop: dict[tuple, list[dict]] = defaultdict(list)
for r in prop_results:
    by_method_prop[group_key(r)].append(r)

by_method_complexity_prop: dict[tuple, list[dict]] = defaultdict(list)
for r in prop_results:
    by_method_complexity_prop[(group_key(r), r['complexity'])].append(r)

GROUPS_PROP = sorted(by_method_prop.keys(), key=lambda k: (k[0], k[1], k[2] or '', k[3]))

print(f'\n{"Method":42s} {"PropAcc(macro)":>14s} {"PropAcc(micro)":>14s} '
      f'{"MissRate(macro)":>15s} {"n":>4s}')
print('-' * 96)

for key in GROUPS_PROP:
    items = by_method_prop[key]
    macro = aggregate_prop_macro(items)
    micro = aggregate_prop_micro(items)

    acc_macro = f'{macro["prop_accuracy_macro"]:.3f}' if macro['prop_accuracy_macro'] is not None else '-'
    acc_micro = f'{micro["prop_accuracy_micro"]:.3f}' if micro['prop_accuracy_micro'] is not None else '-'
    mr_macro  = f'{macro["missing_rate_macro"]:.3f}'  if macro['missing_rate_macro']  is not None else '-'

    print(f'{group_label(key):42s} {acc_macro:>14s} {acc_micro:>14s} {mr_macro:>15s} {macro["n"]:4d}')


Method                                     PropAcc(macro) PropAcc(micro) MissRate(macro)    n
------------------------------------------------------------------------------------------------
b1_claude-sonnet-5_few_shot                         0.612          0.583           0.266   30
b1_claude-sonnet-5_zero_shot                        0.446          0.403           0.444   30
b1_gemini-3.1-pro-preview_few_shot                  0.639          0.567           0.296   30
b1_gemini-3.1-pro-preview_zero_shot                 0.447          0.412           0.446   30
b1_gpt-5.6-terra_few_shot                           0.508          0.430           0.275   30
b1_gpt-5.6-terra_zero_shot                          0.386          0.383           0.467   30
b2_claude-sonnet-5_few_shot                         0.620          0.586           0.250   30
b2_claude-sonnet-5_zero_shot                        0.448          0.410           0.432   30
b2_gemini-3.1-pro-preview_few_shot                  0.58

In [9]:
EVALUATIONS_DIR = Path(f'evaluations/{TYPE}')
EVALUATIONS_DIR.mkdir(parents=True, exist_ok=True)

# 1) Long format: one row per mockup x configuration
BY_MOCKUP_CSV = EVALUATIONS_DIR / f'eval_prop_accuracy_{TYPE}_by_mockup.csv'
BY_MOCKUP_FIELDNAMES = [
    'mockup', 'complexity', 'index', 'approach', 'strategy', 'prompt_strategy', 'model', 'run',
    'prop_accuracy', 'missing_rate', 'correct', 'wrong', 'hallucinated', 'missing', 'unresolved',
    'error_components',
]

with open(BY_MOCKUP_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=BY_MOCKUP_FIELDNAMES)
    writer.writeheader()
    writer.writerows(prop_results)

print(f'Saved: {BY_MOCKUP_CSV}  ({len(prop_results)} rows)')

# 2) Summary: Macro- and Micro-Aggregation by Approach/Strategy/Model/Prompt Strategy
SUMMARY_CSV = EVALUATIONS_DIR / f'eval_prop_accuracy_{TYPE}_summary.csv'
SUMMARY_FIELDNAMES = [
    'approach', 'strategy', 'prompt_strategy', 'model',
    'prop_accuracy_macro', 'prop_accuracy_micro',
    'missing_rate_macro', 'missing_rate_micro',
    'n', 'n_resolved_acc', 'n_resolved_mr',
]

summary_rows = []
for key in GROUPS_PROP:
    approach, strategy, model, prompt_strategy = key
    items = by_method_prop[key]

    macro = aggregate_prop_macro(items)
    micro = aggregate_prop_micro(items)

    summary_rows.append({
        'approach':        approach,
        'strategy':        strategy,
        'prompt_strategy': prompt_strategy,
        'model':           model or '',
        'prop_accuracy_macro': macro['prop_accuracy_macro'],
        'prop_accuracy_micro': micro['prop_accuracy_micro'],
        'missing_rate_macro':  macro['missing_rate_macro'],
        'missing_rate_micro':  micro['missing_rate_micro'],
        'n':               macro['n'],
        'n_resolved_acc':  macro['n_resolved_acc'],
        'n_resolved_mr':   macro['n_resolved_mr'],
    })

with open(SUMMARY_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=SUMMARY_FIELDNAMES)
    writer.writeheader()
    writer.writerows(summary_rows)

print(f'Saved: {SUMMARY_CSV}  ({len(summary_rows)} rows)')

# 3) Per configuration x complexity level (basis for UF4 degradation factor / UF5)
BY_COMPLEXITY_CSV = EVALUATIONS_DIR / f'eval_prop_accuracy_{TYPE}_by_complexity.csv'
BY_COMPLEXITY_FIELDNAMES = [
    'approach', 'strategy', 'prompt_strategy', 'model', 'complexity',
    'prop_accuracy_macro', 'prop_accuracy_micro',
    'missing_rate_macro', 'missing_rate_micro', 'n',
]

by_complexity_rows = []
for key in GROUPS_PROP:
    approach, strategy, model, prompt_strategy = key

    for complexity in COMPLEXITIES:
        items = by_method_complexity_prop[(key, complexity)]

        if not items:
            continue

        macro = aggregate_prop_macro(items)
        micro = aggregate_prop_micro(items)

        by_complexity_rows.append({
            'approach':        approach,
            'strategy':        strategy,
            'prompt_strategy': prompt_strategy,
            'model':           model or '',
            'complexity':      complexity,
            'prop_accuracy_macro': macro['prop_accuracy_macro'],
            'prop_accuracy_micro': micro['prop_accuracy_micro'],
            'missing_rate_macro':  macro['missing_rate_macro'],
            'missing_rate_micro':  micro['missing_rate_micro'],
            'n': macro['n'],
        })

with open(BY_COMPLEXITY_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=BY_COMPLEXITY_FIELDNAMES)
    writer.writeheader()
    writer.writerows(by_complexity_rows)

print(f'Saved: {BY_COMPLEXITY_CSV}  ({len(by_complexity_rows)} rows)')

# 4) Degradation factor (medium/simple, hard/simple) per configuration (UF4)
DEGRADATION_CSV = EVALUATIONS_DIR / f'eval_prop_accuracy_{TYPE}_degradation.csv'
DEGRADATION_FIELDNAMES = [
    'approach', 'strategy', 'prompt_strategy', 'model',
    'prop_accuracy_simple', 'prop_accuracy_medium', 'prop_accuracy_hard',
    'degradation_factor_medium', 'degradation_factor_hard',
]

def degradation_factor(base: float | None, target: float | None) -> float | None:
    if base is None or target is None or base == 0:
        return None
    return round(target / base, 4)


degradation_rows = []
for key in GROUPS_PROP:
    approach, strategy, model, prompt_strategy = key

    simple_items = by_method_complexity_prop[(key, 'simple')]
    medium_items = by_method_complexity_prop[(key, 'medium')]
    hard_items   = by_method_complexity_prop[(key, 'hard')]

    simple_acc = aggregate_prop_macro(simple_items)['prop_accuracy_macro'] if simple_items else None
    medium_acc = aggregate_prop_macro(medium_items)['prop_accuracy_macro'] if medium_items else None
    hard_acc   = aggregate_prop_macro(hard_items)['prop_accuracy_macro']   if hard_items   else None

    degradation_rows.append({
        'approach':        approach,
        'strategy':        strategy,
        'prompt_strategy': prompt_strategy,
        'model':           model or '',
        'prop_accuracy_simple': simple_acc,
        'prop_accuracy_medium': medium_acc,
        'prop_accuracy_hard':   hard_acc,
        'degradation_factor_medium': degradation_factor(simple_acc, medium_acc),
        'degradation_factor_hard':   degradation_factor(simple_acc, hard_acc),
    })

with open(DEGRADATION_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=DEGRADATION_FIELDNAMES)
    writer.writeheader()
    writer.writerows(degradation_rows)

print(f'Saved: {DEGRADATION_CSV}  ({len(degradation_rows)} rows)')

Saved: evaluations\components\eval_prop_accuracy_components_by_mockup.csv  (1620 rows)
Saved: evaluations\components\eval_prop_accuracy_components_summary.csv  (54 rows)
Saved: evaluations\components\eval_prop_accuracy_components_by_complexity.csv  (162 rows)
Saved: evaluations\components\eval_prop_accuracy_components_degradation.csv  (54 rows)
